# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelkareemahmed/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a Binary Classification task combined with Ranking. We need to classify whether a page is 'at risk of decline' (1) or 'stable' (0). After classification, we will output a probability score to rank the pages. This ensures the content team can prioritize their limited time on the most critical pages first.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import subprocess
import sys

# 1. Download the dataset into the Colab environment
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# 2. Now load the data and set up our task
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print("Task Type Defined: Binary Classification & Ranking")


Task Type Defined: Binary Classification & Ranking


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Our proxy target is target_is_at_risk. Since we cannot predict Google's exact future algorithm updates, we use an observed rule based on current signals: pages where the trend_direction is already showing a 'down' pattern AND they have enough visibility (e.g., impressions >= 100) to actually matter to the business.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
# Creating the proxy target based on the observed rule
df['target_is_at_risk'] = np.where(
    (df['trend_direction'].str.lower() == 'down') & (df['impressions_90d'] >= 100),
    1, 0
)
print("Proxy label distribution (0 = Stable, 1 = At Risk):")
print(df['target_is_at_risk'].value_counts())


Proxy label distribution (0 = Stable, 1 = At Risk):
target_is_at_risk
0    16848
1    13152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric is Precision@K (e.g., Precision@50). The content team has limited bandwidth and can only update a few pages a week. We care far more about the top 50 flagged pages being genuinely at risk (high precision) than we care about finding every single declining page in the database (recall). A "good" number means the vast majority of our top recommendations are true positives.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The metric focuses on the top of the ranked list
print("Primary Metric: Precision@K")
print("Business Goal: Maximize the percentage of true 'at-risk' pages in the top K ranked recommendations to save human review time.")


Primary Metric: Precision@K
Business Goal: Maximize the percentage of true 'at-risk' pages in the top K ranked recommendations to save human review time.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is a single webpage (URL). Each row in the dataframe represents the historical performance metrics, traffic data, and metadata for one specific page.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the unit of analysis (One row = One Page)
print("Unit of analysis: One row = One specific webpage")
display(df[['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'target_is_at_risk']].head(3))

Unit of analysis: One row = One specific webpage


,impressions_90d,avg_position,ctr,days_since_last_update,target_is_at_risk
0,3803,10.6,0.76,20,1
1,15320,20.3,0.05,25,1
2,12581,36.5,0.09,20,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like if days_since_last_update > 180 and impressions_90d > 500) is too rigid and highly reactive. It only catches pages after they have severely decayed. The pattern of decline is messy and non-linear; it involves subtle, combined shifts in CTR, average position, and content age. ML can learn these complex interactions to spot early warning signs before a catastrophic drop becomes permanent.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the complexity and variance of the data that an if-statement would struggle to capture accurately
print("Sample of features ML can combine non-linearly to predict risk:")
display(df[['content_age_days', 'days_since_last_update', 'avg_position', 'ctr']].describe().round(2))

Sample of features ML can combine non-linearly to predict risk:


,content_age_days,days_since_last_update,avg_position,ctr
count,30000.00,30000.00,30000.00,30000.00
mean,256.17,46.10,16.34,0.51
std,132.71,42.08,15.22,3.28
min,90.00,1.00,0.00,0.00
25%,132.00,20.00,6.20,0.00
50%,236.00,20.00,10.80,0.07
75%,333.00,104.00,22.30,0.29
max,564.00,373.00,245.00,100.00


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.